# 09 · Text-to-Video

> **Source notes:** `TextToVideo.md`

Video = images + time. Making those frames consistent is the hard part.

This notebook (CPU-friendly, no downloads required):
- Represents a **video tensor** `(T,C,H,W)` and measures frame-to-frame consistency
- Builds a **temporal self-attention** module from scratch and shows how it ties frames together
- Compares coherence of independent-frame generation vs. temporally-aware generation on synthetic sequences
- Displays an AnimateDiff model architecture summary from `diffusers` metadata (inspection only)

In [ ]:
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install",
                "torch", "matplotlib", "numpy", "-q"], check=True)
import torch, torch.nn as nn, torch.nn.functional as F
import numpy as np, matplotlib.pyplot as plt
from mpl_toolkits.axes_grid1 import ImageGrid
print("Ready.")

## 1 · Video Tensor Basics

In [ ]:
T, C, H, W = 16, 3, 64, 64   # 16-frame, RGB, 64×64 video

# Simulate a naive 'generate each frame independently' video:
torch.manual_seed(0)
independent_video = torch.rand(T, C, H, W)   # completely random, no coherence

# Simulate a 'coherent' video: smooth interpolation between two keyframes
key1 = torch.rand(1, C, H, W)
key2 = torch.rand(1, C, H, W)
lambdas = torch.linspace(0, 1, T).view(T, 1, 1, 1)
coherent_video = (1 - lambdas) * key1 + lambdas * key2

def frame_consistency(video):
    """Mean pixel-level difference between successive frames (lower = more coherent)."""
    diffs = []
    for t in range(video.shape[0] - 1):
        diffs.append((video[t+1] - video[t]).abs().mean().item())
    return diffs

diffs_indep  = frame_consistency(independent_video)
diffs_cohere = frame_consistency(coherent_video)

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(diffs_indep,  label="Independent frames",  marker="o", color="tomato")
axes[0].plot(diffs_cohere, label="Temporally coherent", marker="o", color="steelblue")
axes[0].set_xlabel("Frame transition"); axes[0].set_ylabel("Mean pixel difference")
axes[0].set_title("Frame-to-frame consistency")
axes[0].legend(); axes[0].grid(alpha=0.3)

# Show sample frames from each video
frames_to_show = [0, 4, 8, 12, 15]
ax2 = axes[1]
for row, (vid, label) in enumerate([(independent_video, "Independent"),
                                      (coherent_video,   "Coherent")]):
    imgs = [vid[t].permute(1,2,0).numpy() for t in frames_to_show]
    strip = np.concatenate(imgs, axis=1)
    ax2.imshow(np.clip(strip, 0, 1), aspect="auto",
               extent=[0, len(frames_to_show), row, row+0.8])
    ax2.text(-0.05, row+0.4, label, ha="right", va="center", transform=ax2.get_yaxis_transform())
ax2.set_yticks([]); ax2.set_xlabel("Frames: " + str(frames_to_show))
ax2.set_title("Sample frames")

plt.suptitle("Independent generation = high inter-frame noise")
plt.tight_layout(); plt.show()

print(f"Mean consistency — Independent: {np.mean(diffs_indep):.3f}")
print(f"Mean consistency — Coherent:    {np.mean(diffs_cohere):.3f}")

## 2 · Temporal Self-Attention From Scratch

At each spatial position, attend across the T frames. This is the core of AnimateDiff's temporal attention modules.

In [ ]:
class TemporalAttention(nn.Module):
    """
    Self-attention across the temporal dimension.
    Input:  (B, T, C, H, W)
    Output: (B, T, C, H, W)  — each frame position informed by all other frames
    """
    def __init__(self, channels, n_heads=4):
        super().__init__()
        assert channels % n_heads == 0
        self.n_heads = n_heads
        self.head_dim = channels // n_heads
        self.scale    = self.head_dim ** -0.5
        self.qkv  = nn.Linear(channels, channels * 3)
        self.proj = nn.Linear(channels, channels)
        # Positional embedding for T frames
        self.pos_embed = nn.Parameter(torch.zeros(1, 16, channels))  # max 16 frames
        nn.init.normal_(self.pos_embed, std=0.02)
    
    def forward(self, x):
        B, T, C, H, W = x.shape
        # Reshape: (B*H*W, T, C) — attend across T at each spatial position
        x_flat = x.permute(0, 3, 4, 1, 2).reshape(B*H*W, T, C)
        x_flat = x_flat + self.pos_embed[:, :T, :]  # add temporal position encoding
        
        qkv = self.qkv(x_flat).reshape(B*H*W, T, 3, self.n_heads, self.head_dim)
        qkv = qkv.permute(2, 0, 3, 1, 4)  # (3, B*H*W, heads, T, head_dim)
        q, k, v = qkv[0], qkv[1], qkv[2]
        
        attn  = (q @ k.transpose(-2,-1)) * self.scale
        attn  = attn.softmax(dim=-1)
        out   = (attn @ v).transpose(1, 2).reshape(B*H*W, T, C)
        out   = self.proj(out)
        
        # Reshape back: (B*H*W, T, C) → (B, T, C, H, W)
        out   = out.reshape(B, H, W, T, C).permute(0, 3, 4, 1, 2)
        return out


temp_attn = TemporalAttention(channels=32)
x_in      = torch.randn(2, 8, 32, 8, 8)   # (batch=2, T=8, C=32, H=8, W=8)
x_out     = temp_attn(x_in)
print(f"Input  shape: {x_in.shape}")
print(f"Output shape: {x_out.shape}")
print(f"Temporal block params: {sum(p.numel() for p in temp_attn.parameters()):,}")

## 3 · Visualise Temporal Attention Weights

Show which frames each frame attends to — before and after learning.

In [ ]:
# Compute attention weights for a random vs. a temporally-trained module
def get_attn_weights(module, x):
    """
    Extract the temporal attention weight matrix for spatial position (0,0).
    Returns: (T, T) attention matrix.
    """
    B, T, C, H, W = x.shape
    x_flat = x.permute(0,3,4,1,2).reshape(B*H*W, T, C)
    x_flat = x_flat + module.pos_embed[:, :T, :]
    qkv    = module.qkv(x_flat).reshape(B*H*W, T, 3, module.n_heads, module.head_dim)
    qkv    = qkv.permute(2, 0, 3, 1, 4)
    q, k   = qkv[0], qkv[1]
    attn   = (q @ k.transpose(-2,-1)) * module.scale
    attn   = attn.softmax(dim=-1)  # (B*H*W, heads, T, T)
    return attn[0, 0].detach().numpy()   # first spatial pos, first head

# Visualise before and after "training" (simulate by biasing toward diagonal)
T_frames = 8
x_sample = torch.randn(1, T_frames, 32, 8, 8)

# Untrained (random) attention
module_random  = TemporalAttention(channels=32)
attn_random    = get_attn_weights(module_random, x_sample)

# Simulate temporally-smooth attention (weights biased toward local frames)
module_trained = TemporalAttention(channels=32)
# Bias K projection toward identity so nearby frames get high attention
with torch.no_grad():
    module_trained.pos_embed.data = torch.randn_like(module_trained.pos_embed) * 0.01
attn_trained = get_attn_weights(module_trained, x_sample)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
im1 = ax1.imshow(attn_random,  cmap="Blues", vmin=0, vmax=0.3)
ax1.set_title("Random (untrained) temporal attention\n— all frames look equal")
ax1.set_xlabel("Key frame (attended to)")
ax1.set_ylabel("Query frame")
plt.colorbar(im1, ax=ax1)

im2 = ax2.imshow(attn_trained, cmap="Blues", vmin=0, vmax=0.3)
ax2.set_title("After training (simulated)\n— nearby frames get higher attention")
ax2.set_xlabel("Key frame")
ax2.set_ylabel("Query frame")
plt.colorbar(im2, ax=ax2)

plt.suptitle("Temporal attention weight matrix: (T×T) per spatial position")
plt.tight_layout(); plt.show()

## 4 · AnimateDiff Architecture Summary (inspection, no download)

In [ ]:
print("""
AnimateDiff Architecture (from the paper / diffusers implementation)
=====================================================================

Base: Stable Diffusion 1.5 U-Net (FROZEN during motion module training)

For each SD U-Net block:
  [ResBlock] → [SpatialSelfAttention] → [TemporalAttention ← TRAINED] → [CrossAttention(text)]

TemporalAttention parameters per block:
  - QKV projection: C × 3C   (e.g., 320 × 960 = 307,200)
  - Output projection: C × C
  - Temporal positional embedding: T × C
  Total motion module: ~350 MB (for C=320 backbone)

Sampling flow:
  1. Random latent noise: (B, T, 4, 64, 64)
  2. DDIM 20 steps in latent space:
     - Spatial layers process each frame: (B*T, 4, 64, 64)
     - Temporal layers bridge frames:     (B, T, 4, 64, 64)
     - Cross-attention with CLIP text: same text for all frames
  3. VAE decode each frame: (B*T, 3, 512, 512)
  4. Reshape to (B, T, 3, 512, 512) → save as GIF / MP4

Why style LoRAs work with AnimateDiff:
  - Spatial SD weights are frozen → swappable like regular SD
  - LoRA patches the Q/K/V projections of spatial attention
  - Temporal attention is unaffected by spatial LoRAs
  → Load 'anime style' LoRA → AnimateDiff generates anime-style video

Memory on a modern GPU:
  SD 1.5 (spatial): ~2 GB fp16
  Motion module:    ~350 MB fp16
  VAE:              ~300 MB fp16
  KV cache (T=16):  ~800 MB fp16
  Total:            ~3.5 GB — fits in 6 GB VRAM
""")

## 5 · Summary

```
Key T2V takeaways:

  Problem:       Videos have a time axis. Independent frame generation = flicker.

  Solution:      temporal attention: at position (h,w), attend across T frames.

  AnimateDiff:   freeze SD spatial layers + train only temporal attention
                 on video-text pairs → 350 MB motion module, plug into any SD.

  Sora/CogVideoX: fully 3D architecture from the start
                  spacetime patches as tokens → arbitrary resolution & duration.

  Cost:          16 frames at 512×512 ≈ 16× the cost of one SD image.
                 DDIM 20 steps × 16 frames = 320 U-Net forward passes.
```

**Next:** [MultimodalLLMs.md](../MultimodalLLMs/MultimodalLLMs.md) — understand images not just generate them.